# Five cities and a nation — crime, enforcement and police strength

The other FBI notebooks are about the plumbing: [`api.ipynb`](api.ipynb) covers
the endpoints and the ways they mislead, [`client.ipynb`](client.ipynb) getting
series out of them, [`mcp.ipynb`](mcp.ipynb) the three tools a model sees. This
one asks a question of the data, and [`enforcement.ipynb`](enforcement.ipynb)
takes the arrest half of it further than a section can.

**The question.** How do crime, enforcement intensity and police strength
compare across five American cities and the nation — and what can honestly be
said across 2021, the year the FBI retired the system that produced every
earlier number?

Four measures answer it, and the FBI is unusual in serving all four from one
API: homicides per 100,000 residents (what happened), the share of homicides
cleared (what the department did about it), arrests per 100,000 residents for
any offence (how often it arrested anyone at all), and sworn officers per
100,000 residents (what it had to do it with). Twenty-nine calls in total —
twelve offence series, six arrest series, five staffing series, and six more in
section 5, which is where the notebook stops describing and starts testing.

Homicide throughout, for one reason. Under the old reporting system an incident
contributed only its most serious offence, and nothing outranks a homicide, so
the rule that was abolished in 2021 never suppressed one. Every other offence
in the catalogue carries a change in *counting* into 2021 on top of everything
else that happened that year. Section 5 goes and gets one of them deliberately.
The arrest total at the end of section 3 is the other exception, and just as
deliberate: only the total over every offence counts how often a department
arrested anyone at all. What the arrest route does one offence at a time, and
what its filing schedule looks like underneath, is
[`enforcement.ipynb`](enforcement.ipynb).


In [22]:
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Anchor on this notebook's own directory rather than on the working directory:
# VS Code can root a notebook at the workspace folder instead of at the file,
# and then a relative "../" resolves somewhere else entirely and the imports
# below fail with ModuleNotFoundError.
_here = Path.cwd()
_FBI = _here if (_here / "utils" / "mcp.py").exists() else _here / "notebooks" / "fbi"
for _path in (str(_FBI), str(_FBI.parents[1])):      # notebooks/fbi, then the meida root
    if _path not in sys.path:
        sys.path.insert(0, _path)

import asyncio
import time
from datetime import datetime
from typing import Any

import numpy
from matplotlib import pyplot
from numpy.typing import NDArray

from lib import config
from utils.mcp import call_tool as _call_tool

pyplot.style.use(config.glyfish_style)

#: Seconds to leave between requests. The hourly allowance is shared across
#: everything using the key, and a burst is answered with "503 upstream connect
#: error" -- which is charged to the allowance as well, and which the client
#: retries four times before giving up. Twenty-nine calls four seconds apart
#: cost two minutes and have not met one.
PACE = 4.0
_last_call = 0.0


async def call_tool(name: str, arguments: dict[str, Any]) -> Any:
    """utils.mcp.call_tool, never less than PACE seconds after the last one."""
    global _last_call
    await asyncio.sleep(max(0.0, PACE - (time.monotonic() - _last_call)))
    try:
        return await _call_tool(name, arguments)
    finally:
        _last_call = time.monotonic()

#: The six scopes, their NIBRS conversion dates, and the colour each keeps
#: throughout. The dates are from the agency registry -- `get_agencies(state)`,
#: walked in client.ipynb section 1 -- rather than from any series, which is
#: what makes section 5 possible: they are an outside fact to test against.
SCOPES: dict[str, dict[str, Any]] = {
    "national":  {"name": "United States", "nibrs": None,         "colour": "0.45"},
    "NY0303000": {"name": "New York City", "nibrs": "2023-01-01", "colour": "#0067C4"},
    "DCMPD0000": {"name": "Washington",    "nibrs": "2021-08-01", "colour": "#FF9500"},
    "CA0194200": {"name": "Los Angeles",   "nibrs": "2024-05-01", "colour": "#5600C9"},
    "CA0380100": {"name": "San Francisco", "nibrs": None,         "colour": "#AC9C00"},
    "TN0190100": {"name": "Nashville",     "nibrs": "1999-10-01", "colour": "#AC6500"},
}
CITIES = [scope for scope in SCOPES if scope != "national"]

START, END = "01-1985", "12-2024"
SRS_END = datetime(2021, 1, 1)      # the day the Summary Reporting System closed


## 1. The six scopes, and why these six

Five city departments and the nation they sit in. The cities are not a sample
of anything — they were chosen because their **conversion dates span
twenty-five years**, which is the only reason section 5 can be attempted.

The FBI retired the Summary Reporting System (SRS) on 1 January 2021 in favour
of the National Incident-Based Reporting System (NIBRS). Agencies did not
convert on that date, though; they converted on their own dates, over two
decades, and the registry records each one. Nashville was reporting under NIBRS
twenty-one years before the changeover. San Francisco has never converted at
all. Washington converted seven months into 2021, New York two years after it,
Los Angeles three.

That spread is the shape of a natural experiment. Sections 2 to 4 mark the
dates on their figures and read nothing off them; section 5 uses them as a
prediction, and [`enforcement.ipynb`](enforcement.ipynb) measures the step in
the arrest count across each of them, where they settle nothing.

(Nashville is here because I live here. It earns its place anyway: it is the
only city of the five whose series crosses 2021 without a hole in it, which
makes it the control the other four need.)


In [23]:
print(f"{'scope':11} {'place':16} {'NIBRS from':12} status at the 2021 changeover")
for scope, meta in SCOPES.items():
    if scope == "national":
        print(f"{scope:11} {meta['name']:16} {'—':12} the sum of whoever reported")
        continue
    if meta["nibrs"] is None:
        status = "never converted — still a summary reporter"
    else:
        converted = datetime.strptime(meta["nibrs"], "%Y-%m-%d")
        months = (converted.year - 2021) * 12 + converted.month - 1
        status = (f"converted {abs(months)} months "
                  f"{'before' if months < 0 else 'after'} SRS closed")
    print(f"{scope:11} {meta['name']:16} {meta['nibrs'] or 'never':12} {status}")

scope       place            NIBRS from   status at the 2021 changeover
national    United States    —            the sum of whoever reported
NY0303000   New York City    2023-01-01   converted 24 months after SRS closed
DCMPD0000   Washington       2021-08-01   converted 7 months after SRS closed
CA0194200   Los Angeles      2024-05-01   converted 40 months after SRS closed
CA0380100   San Francisco    never        never converted — still a summary reporter
TN0190100   Nashville        1999-10-01   converted 255 months before SRS closed


## The twelve calls

`fbi_offenses` takes a scope, an offence and a measure, so offences and
clearances for six scopes are twelve calls. Each response carries the series
plus the two things needed to read it: the **population** it happened in and
the **coverage** percentage — the share of the population living where agencies
reported.

The window stops at December 2024 so that every scope contributes whole years
and so the figures match the other three notebooks; the vintage printed below
shows the programme has filed beyond it since.

Coverage deserves a squint at the agency grain. The CDE publishes it per
jurisdiction, and a city department's payload carries its *state's* figure
where it has none of its own; the response says so in a notice rather than
passing one off as the other. It matters here, because "California, 20.6%
covered in February 2021" says nothing whatever about whether the LAPD filed
that month. Not one of the five gets its own figure. Washington's comes closest
— the District is one jurisdiction with one police force, which is why it reads
100% in every month of forty years, including the seven in 2021 when the
department filed nothing at all.

In [24]:
series: dict[str, dict[str, Any]] = {}
for scope in SCOPES:
    for measure in ("offenses", "clearances"):
        series[f"{scope}/{measure}"] = await call_tool("fbi_offenses", {
            "offense": "homicide", "scope": scope, "measure": measure,
            "unit": "count", "start": START, "end": END,
        })

print(f"{'place':16} {'geography, as the CDE labels it':42} {'coverage':>9} {'worst':>8}"
      f"  whose coverage")
for scope, meta in SCOPES.items():
    payload = series[f"{scope}/offenses"]
    label = payload["coverage"]["label"]
    print(f"{meta['name']:16} {payload['geography'][:42]:42} "
          f"{payload['coverage_mean_percent']:8.1f}% {payload['coverage_min_percent']:7.1f}%"
          f"  {label}"
          f"{'' if label == payload['geography'] else ' ← not this department'}")

print(f"\nvintage {series['national/offenses']['max_data_date']}, refreshed "
      f"{series['national/offenses']['last_refresh_date']}, "
      f"{len(series['national/offenses']['observations'])} monthly periods per series")

RuntimeError: fbi_offenses returned no structured content: 'Error executing tool fbi_offenses: FBI request failed (503): upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote connection failure'

## 2. Homicide per 100,000 — and the filing schedule underneath it

Before anything can be plotted there is a problem in the grain, which
[`client.ipynb`](client.ipynb) section 3 sets out in full: **a month with no
return reports as a number.** Every one of the 480 periods above carries a
value and almost none is null, yet New York filed quarterly from 2002 to 2012 —
eight zeros a year, each quarter landing whole in its third month — and
Washington filed twice in 1996 and once in 1997.

Three ways a period fails, all of them present in these six series, and each
needing its own test.

*A run of blanks.* Zeros or nulls, two or more in a row, in a series where
enough was expected to fall in them. The two spellings are the same fact: New
York and Washington report **zero** for the months of 2021 they did not file,
Los Angeles and San Francisco report **null** for theirs, and nothing in the
payload distinguishes them. The length on its own is not enough, which is what
the floor at the end of this section is for.

*A filing that is not one.* New York's aggravated assault series for 2021 reads
1, 2, 4, 1, 6, 3, 4, 0, 3, 4, 2, 4 against a normal month of about 2,900. No
test based on zeros catches that; a test on the level does.

*A catch-up.* The filing immediately after a gap may be carrying the gap —
Washington's December 1997 holds the whole year at 301 against a normal 22.
Here the evidence decides rather than the position: a January that carries a
year is far larger than a January, and one that does not, is not. That is the
one departure from `client.ipynb`'s rule, which suspects every filing after a
gap. It matters at exactly one place — January 2022, ordinary in all three
cities that lost 2021 — and that is the month the whole 2021 question turns on.

Where the counts are small no test can work, and both of the first two carry a
floor because of it. San Francisco records about four homicides a month and
clears two or three, so a zero is an ordinary month there and two in a row an
ordinary quarter: a run is only read as a gap where the filings it swallowed —
its length times the local level — amount to something. Set the floor and
several of San Francisco's clearance years come back, section 3 says which. The
price is exactly symmetric and worth saying out loud: a short genuine gap in a
small series now reads as a quiet stretch, and nothing in the payload can tell
the two apart.

The rule itself is not written out here. It lives in
[`utils/filing.py`](utils/filing.py), because
[`enforcement.ipynb`](enforcement.ipynb) applies the same one to the arrest
series and a second copy would drift from this one; the next cell prints it as
the module defines it.


In [ ]:
# The filing rule is one implementation, in utils/filing.py, because
# enforcement.ipynb applies the same one to the arrest series and a second copy
# would drift from this one. Its docstrings carry the reasoning -- which
# department each floor is for, and what it costs -- and its decisions are
# pinned by tests/test_fbi_notebook_filing.py. The older, simpler unfiled() and
# rolling_12() in utils/mcp.py are the first version of the same idea, kept
# because mcp.ipynb walks the tools with them; the module docstring says what
# they lack.
import inspect

from utils.filing import (Payload, arrest_unreadable, context, local_median, monthly,
                          stamp_source, trailing_sum, unreadable)

# Printed rather than restated, so the thresholds on this page are the ones
# that ran: each function's signature with its defaults, and the first line of
# what it decides.
for _function in (monthly, context, local_median, unreadable, trailing_sum,
                  arrest_unreadable):
    _defaults = ", ".join(
        f"{_name}={_parameter.default:.3g}"
        for _name, _parameter in inspect.signature(_function).parameters.items()
        if _parameter.default is not inspect.Parameter.empty)
    print(f"{_function.__name__}({_defaults})")
    print(f"    {_function.__doc__.splitlines()[0]}")


One class holds a scope's three derived series, so that the rest of the
notebook reads as comparisons rather than as arithmetic. Two decisions are
baked into it.

**The rate is computed here, not requested.** `fbi_offenses` will return
`unit='rate'`, and for an agency the CDE divides by that agency's own
population base, which is not always the city. The population arrives monthly
on the same response, so a rate built from counts and that denominator is one
whose arithmetic is visible.

**The clearance ratio drops a window if either leg is unreadable.** Both legs
of a return are filed together, so a gap in one is usually a gap in both; but
the count only needs its own leg, and there is no reason to throw away
Washington's homicides because its clearances were filed late.

In [ ]:
class Place:
    """One scope's homicide series, cleaned of its filing schedule."""

    def __init__(self, scope: str, offences: Payload, clearances: Payload) -> None:
        self.scope, self.meta = scope, SCOPES[scope]
        self.dates, counts = monthly(offences)
        _, cleared = monthly(clearances)
        self.population = context(offences, "population", self.dates)
        self.coverage = context(offences, "coverage", self.dates)

        self.gaps = unreadable(counts)
        either = self.gaps | unreadable(cleared)
        self.counts = numpy.where(self.gaps, numpy.nan, counts)
        self.offences = trailing_sum(self.counts)
        self.rate = 1e5 * self.offences / self.population
        self.clearance = 100.0 * (trailing_sum(numpy.where(either, numpy.nan, cleared))
                                  / trailing_sum(numpy.where(either, numpy.nan, counts)))

    def at(self, values: NDArray[numpy.float64], period: str) -> float:
        """One month's value of a derived series, by 'MM-YYYY'."""
        month, year = (int(part) for part in period.split("-"))
        index = numpy.flatnonzero([date.year == year and date.month == month
                                   for date in self.dates])
        return float(values[index[0]]) if len(index) else float("nan")

    def annually(self, values: NDArray[numpy.float64]
                 ) -> tuple[list[datetime], NDArray[numpy.float64]]:
        """One point a year: the value of a trailing series at each December."""
        index = [i for i, date in enumerate(self.dates) if date.month == 12]
        return [self.dates[i] for i in index], values[index]

    def holes(self) -> list[str]:
        """The unreadable stretches, as 'YYYY-MM' or 'YYYY-MM..YYYY-MM'."""
        spans, start = [], None
        for date, gap in zip(self.dates, self.gaps):
            if gap and start is None:
                start = date
            elif not gap and start is not None:
                previous = self.dates[numpy.flatnonzero(self.dates == date)[0] - 1]
                spans.append(f"{start:%Y-%m}" if start == previous
                             else f"{start:%Y-%m}..{previous:%Y-%m}")
                start = None
        return spans


places = {scope: Place(scope, series[f"{scope}/offenses"], series[f"{scope}/clearances"])
          for scope in SCOPES}

print(f"{'place':16} {'unreadable':>10} {'2021 filed':>11}  where the holes are")
for place in places.values():
    filed = sum(1 for date, gap in zip(place.dates, place.gaps)
                if date.year == 2021 and not gap)
    spans = place.holes()
    shown = spans if len(spans) <= 4 else spans[:2] + [f"… {len(spans) - 3} more"] + spans[-1:]
    print(f"{place.meta['name']:16} {int(place.gaps.sum()):10d} {filed:8d}/12  "
          f"{', '.join(shown) if spans else 'none'}")

Washington's hole closes in the month it converted. New York's forty stretches
are its quarterly decade plus the whole of 2021. Nashville has no hole at all.

Now the picture. The vertical scale is logarithmic, which is not decoration:
Washington's 1991 peak is nine times the national rate of the same month and
twenty-four times New York's 2017 floor, and drawn linearly five of the six
lines are a flat worm along the bottom. On a log scale equal vertical distances
are equal *proportional* changes, which is the comparison the question asks
for.

In [ ]:
figure, axis = pyplot.subplots(figsize=(12, 6.5))
axis.set_axisbelow(True)
axis.axvspan(SRS_END, datetime(2022, 1, 1), color="0.55", alpha=0.25, lw=0)
axis.text(datetime(2021, 7, 1), 0.02, "2021", transform=axis.get_xaxis_transform(),
          ha="center", va="bottom", fontsize=8, alpha=0.85)

for scope, place in places.items():
    # Nashville heavier than the other four: it is the city this is written
    # from, and the one closest in colour to San Francisco.
    axis.plot(place.dates, place.rate, color=place.meta["colour"],
              lw={"national": 3.0, "TN0190100": 2.4}.get(scope, 1.6),
              label=place.meta["name"], zorder=2 if scope == "national" else 3)
    # Each department's conversion marked on its own line rather than in the
    # legend: section 5 asks whether anything happens there.
    if place.meta["nibrs"]:
        converted = datetime.strptime(place.meta["nibrs"], "%Y-%m-%d")
        value = place.at(place.rate, f"{converted.month:02d}-{converted.year}")
        if numpy.isfinite(value):
            axis.plot([converted], [value], marker="o", markersize=8, mfc="none",
                      mew=1.8, color=place.meta["colour"], zorder=4)

axis.set_yscale("log")
axis.set_yticks([2, 5, 10, 20, 50, 100])
axis.set_yticklabels(["2", "5", "10", "20", "50", "100"])
axis.set_ylim(2, 130)
axis.set_ylabel("Homicides per 100,000 residents, twelve months to date")
axis.set_xlabel("Year")
axis.set_title("Homicide rate, trailing twelve months, 1986–2024 "
               "(rings mark each department's NIBRS conversion)", fontsize=12)
axis.legend(loc="upper right", ncols=3, fontsize=9)
figure.tight_layout()
stamp_source("fbi_offenses(homicide, {national, NY0303000, DCMPD0000, CA0194200, "
             "CA0380100, TN0190100}, offenses, count)",
             "rate = 1e5 × trailing twelve-month count ÷ the population on the same response",
             figure=figure)

Five things are in that picture.

**The arc is shared; the levels are not.** Every line peaks between 1990 and
1993, falls for twenty years, bottoms out around 2013 and turns up after 2019 —
the same shape at rates an order of magnitude apart. Washington peaks near 81
per 100,000 in September 1991, against a national peak of 9.5 three months
later, and is still the highest of the six in 2024.

**Nashville is the exception to the arc, twice.** It peaks in November 1997,
four to six years after the other four, and its rise since 2013 is the steepest of
the six: 5.5 per 100,000 in 2013 to 14.7 in 2024, while New York went 4.0 to
4.3. In 1990 it had the lowest homicide rate of the five cities; in 2024 it has
the second highest.

**The 2021 band is mostly empty.** Four of the five cities have no line through
it, because they filed nothing that year. The absence is the finding, not a
failure of the drawing.

**The national line dips in 2021** — the only line that does, which should be
suspicious. It is those same missing cities, aggregated: the numerator lost the
agencies that had not converted while the denominator stayed the whole
population. The next cell takes the dip apart.

**The rings sit in unremarkable places.** Each marks the month a department
converted, on its own line, and none of them is where the line does anything.
Washington has no ring at all: its rate is unreadable in the month it
converted, which is the 2021 hole seen from the inside.

**New York's line stops for eleven years.** That is a department filing four
times a year: the annual totals for 2002–2012 are recoverable, because each
quarter lands whole inside its own year, but no twelve-month window ending in a
month other than March, June, September or December is, and the rule does not
carve out the four that are.

In [ ]:
print("Homicides per 100,000, twelve months to December")
print(f"{'place':16}" + "".join(f"{year:>9}" for year in (1990, 2000, 2013, 2019, 2022, 2024)))
for place in places.values():
    print(f"{place.meta['name']:16}" + "".join(
        f"{place.at(place.rate, f'12-{year}'):9.1f}"
        if numpy.isfinite(place.at(place.rate, f"12-{year}")) else f"{'—':>9}"
        for year in (1990, 2000, 2013, 2019, 2022, 2024)))

# The national dip, arithmetically. Dividing by the population that reported
# rather than by everybody assumes homicide is distributed the same way inside
# and outside the reporting share -- an assumption, not a fact, but a much
# smaller one than the published rate makes.
nation = places["national"]
print("\nThe United States line against the coverage it rests on")
print(f"{'year':6}{'coverage':>10}{'as published':>14}{'over the reporting population':>32}")
for year in range(2018, 2025):
    published = nation.at(nation.rate, f"12-{year}")
    covered = float(numpy.nanmean([nation.coverage[i] for i, date in enumerate(nation.dates)
                                   if date.year == year]))
    print(f"{year:<6}{covered:9.1f}%{published:14.1f}{100 * published / covered:32.1f}")

The dip goes away. Nationally 2021 reads 5.3 homicides per 100,000 as
published, sitting between 6.5 in 2020 and 6.5 in 2022; over the population
that actually reported it reads 6.9, between 6.8 and 6.9. **2021 was not a
quiet year, it was a badly observed one** — and the coverage field, carried on
every response, is the whole of what turns the first reading into the second.

## 3. Clearance rates — what a clearance is, and what it is not

A **clearance** is the FBI's word for an offence the department has closed, and
it happens in one of two ways: an arrest where the case is referred for
prosecution, or an *exceptional* clearance, where the offender is identified
and located but something outside the department's control prevents the arrest
— the suspect is dead, or in custody elsewhere and not extraditable, or the
victim will not cooperate. Clearances ÷ offences is the standard measure of
enforcement intensity and needs no second source: both legs arrive in the same
payload.

Three consequences, all visible below.

**It is not a conviction rate.** Nothing in a clearance says a court agreed. It
is the police account of a case being closed.

**It is not a solve rate for the offences in the window, either.** A clearance
is recorded in the month it happens, not in the month of the offence it closes.
A department working old cases clears more homicides than it recorded — San
Francisco's 94% for 2024 is 33 clearances against 35 homicides, and some of
those 33 belong to earlier years. Four of these six pass 100% at some point.
Read it as closing *activity*.

**A monthly ratio is meaningless.** Three homicides and five clearances in a
month is 167%, which says nothing at all. Hence the trailing year, and hence
dropping any window where either leg has a gap in it. Even annually it happens:
Los Angeles closed 108% of the homicides it recorded in the year to December
2013.

In [ ]:
print("Homicide clearance rate, %, twelve months to December")
print(f"{'place':16}" + "".join(f"{year:>9}" for year in (1990, 2000, 2016, 2019, 2022, 2024)))
for place in places.values():
    print(f"{place.meta['name']:16}" + "".join(
        f"{place.at(place.clearance, f'12-{year}'):9.1f}"
        if numpy.isfinite(place.at(place.clearance, f"12-{year}")) else f"{'—':>9}"
        for year in (1990, 2000, 2016, 2019, 2022, 2024)))

print(f"\n{'place':16} {'any window, %':>13} {'worst window':>16} {'Decembers, %':>15}")
for place in places.values():
    drawn = numpy.flatnonzero(numpy.isfinite(place.clearance))
    worst = int(drawn[numpy.argmin(place.clearance[drawn])])
    _, annual_rate = place.annually(place.clearance)
    print(f"{place.meta['name']:16} "
          f"{numpy.nanmin(place.clearance):5.0f}–{numpy.nanmax(place.clearance):<6.0f} "
          f"{place.clearance[worst]:8.0f}% to {place.dates[worst]:%m-%Y} "
          f"{numpy.nanmin(annual_rate):6.0f}–{numpy.nanmax(annual_rate):<6.0f}")

Nothing here is a steady institutional constant. Washington closed 56% of its
homicides in the year to December 1990, 31% in 2000, 69% in 2016 and 45% in
2024 — a range wider than the difference between any two cities in any one
year. Los Angeles runs 37% in 2000 and 76% in 2019. The national figure moves
between 50% and 70% across the whole period and is the steadiest of the six,
which is what aggregation does.

**Its steadiest is also its lowest, and its lowest is on the break.** The worst
twelve months in forty years of the national series are the twelve to December
2021 — 50%, 49.8 before rounding, the one window that lies wholly inside the
coverage trough. The slide starts earlier, 51.7% in the year to December 2020,
but it does not lift until 2021 is out of the window: every one of the
twenty-three windows touching that year sits between 49.8 and 51.2%, against
55.2% in the year to December 2019 and 57.3% in 2023. Section 2 took the
composition effect out of the national *rate* by dividing by the population
that reported. A ratio has no denominator to rescale, so the same effect is
still standing here: what the numerator and denominator both lost in 2021 was
not a share of the population but a particular set of departments — four of the
five in this notebook among them — and there is no reason their clearance rates
matched the ones left behind. Whether the 50% is enforcement in a hard year or
the composition of who was still filing, this series cannot say. The national
line is least trustworthy at the point it looks most stable.

**New York's clearance leg is out for two years, not one.** It files no
clearance in any month of 2021 *or* 2022 — twenty-four zeros — while its
homicide counts resume in January 2022 and read normally, 31 and 36 and 35.
Both legs are needed for a ratio, so that is the em-dash in the 2022 column
above, and in the left panel of section 4's figure it is a hole running from
December 2020 to December 2023: no twelve-month window in between has both of
its legs. The first clearance New York files after the gap is January 2023, ten
of them — and that date is section 5's business.

**San Francisco is the small-count floor, in both directions.** Its 2000 now
reads 47.5% — 28 clearances against 59 homicides, every month of it filed —
because the two blank months that used to void the year are two ordinary months
in a series that runs two and three. Four more Decembers come back with it:
1996 at 57%, 2007 at 48%, 2011 at 36% and 2008 at 17%, which is 17 clearances
against 98 homicides with five zero months scattered through the year. That
last one drags the worst window in the series down to 13% to September 2008.
Whether 2008 is a department that stopped closing cases or one that stopped
filing the ones it closed is exactly what a series this small cannot tell you.
The number is drawn because the rule cannot justify dropping it, not because it
is trusted.

The one movement that lines up with a system change is Washington's fall from
69% to the low forties, which begins within months of its NIBRS conversion and
its seven unfiled months. How much of that is enforcement and how much is the
changeover is not something this series can answer — and that question is
section 5.

### Arrests — what an arrest is, and what it is not

A clearance, above, is either an arrest or an exceptional clearance, and
`fbi_offenses` reports only the sum. `fbi_arrests` looks as though it might
supply the missing half. It does not, and the reasons are in the definitions
rather than in the data. The response brings none of its own: the API has no
dictionary, as [`api.ipynb`](api.ipynb) notes, and the arrest payload behind the
tool labels its numbers "Arrests" and "Arrests per 100,000 people" and says
nothing more. What follows is the FBI's, from *Crime in the United States,
2019* and the *2019 NIBRS User Manual*.

**What it is.** A count a month per geography of arrests filed — for every
offence together, or, through the tool's `offense` argument, for one of 47 UCR
arrest codes. Its unit is an occasion, not a person: the programme "counts one
arrest for each separate instance in which a person is arrested, cited, or
summoned for an offense", so that "the UCR arrest figures do not reflect the
number of individuals who have been arrested; rather, the arrest data show the
number of times that persons are arrested"
([Persons Arrested](https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/topic-pages/persons-arrested)).

**What it is not: the arrest half of the clearances.** Three reasons. The first
and the third hold for a single offence as much as for the total; the second is
the total's alone.

*Different units.* A clearance counts an offence closed; an arrest counts a
person arrested on one occasion. "In its clearance calculations, the UCR Program
counts the number of offenses that are cleared, not the number of persons
arrested. The arrest of one person may clear several crimes, and the arrest of
many persons may clear only one offense"
([Clearances](https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/topic-pages/clearances)).

*Different coverage.* Clearances exist here only for the offences
`fbi_offenses` serves — eight of them, and two aggregates. The total runs across
every offence, and the FBI's own arrest tables run on past the eight through
other assaults, vandalism, drug abuse violations, driving under the influence
and disorderly conduct, among others, none of which has a clearance series here
at all
([Table 69](https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/tables/table-69)).
The next cell puts a size on it. Narrowing the arrests to one offence removes
this difference, and neither of the other two.

*An exceptional clearance involves no arrest.* It is recorded when "elements
beyond law enforcement's control prevent the agency from arresting and formally
charging the offender" (Clearances, as above), and under incident-based
reporting an agency "cannot clear an incident exceptionally if it was previously
or is concurrently cleared by arrest"
([NIBRS User Manual](https://ucr.fbi.gov/nibrs/nibrs-user-manual), Data
Element 4).

So arrests and clearances neither difference nor bound each other, in total or
for one offence: where one arrest can clear several offences and several arrests
can clear one, the gap between the two counts measures nothing either definition
names. The tool's first notice says the same. Nor is the split served here. The
Summary Reporting System did not record it — "The UCR Program does not
distinguish between offenses cleared by arrest and those cleared by exceptional
means in collecting or publishing data via the traditional Summary Reporting
System" (Clearances, as above). NIBRS does, as a field of its own (Data
Element 4, Cleared Exceptionally), and **none of the three tools this notebook
calls reads that field**: `fbi_offenses` serves one clearance count whichever
system a month was filed under. The field is in the incident files the FBI
publishes for download, which are not an API at all, and
[`enforcement.ipynb`](enforcement.ipynb) section 2 reads them for New York and
Nashville.

What the series *can* answer is its own question: how often a department
arrested anyone, for anything, per resident. Six calls, one per scope, with the
rate computed from the counts and the population on the same response, as in
section 2, and the filing rule of section 2 with the three additions an arrest
series needs. Everything past that is the other notebook's: six offence codes at
these six scopes, the three additions and the margin between each threshold and
the data, each department's filing schedule month by month, and what a step at a
conversion date is worth.


In [ ]:
#: What an all-offence arrest total is made of, in the FBI's own figures: the
#: national estimates on the "Persons Arrested" page of Crime in the United
#: States, 2019, drawn from its Table 29. They are estimates for the whole
#: country, not a CDE series -- here to say what the total contains, not to be
#: set against it month by month.
#: https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/topic-pages/persons-arrested
CIUS_2019: dict[str, int] = {
    "all offences": 10_085_207,
    "violent crime": 495_871,             # murder, rape, robbery, aggravated assault
    "property crime": 1_074_367,          # burglary, larceny-theft, motor vehicle theft, arson
    "drug abuse violations": 1_558_862,
    "driving under the influence": 1_024_508,
}

arrests: dict[str, Payload] = {}
for scope in SCOPES:
    arrests[scope] = await call_tool("fbi_arrests", {
        "scope": scope, "offense": "all", "unit": "count", "start": START, "end": END})


class ArrestSeries:
    """One scope's all-offence arrests, cleaned of its filing schedule.

    The rule is section 2's, with the three additions an arrest series needs
    and a homicide series does not -- a token read as a blank, a part-filing
    line at a third rather than a tenth, and a lump measured anywhere rather
    than only after a gap. enforcement.ipynb section 3 sets them out, prints
    the months each takes, and measures how near each threshold comes to the
    data; utils/filing.py carries them.
    """

    def __init__(self, scope: str, payload: Payload) -> None:
        self.scope, self.meta = scope, SCOPES[scope]
        self.dates, self.filed = monthly(payload)
        self.gaps = arrest_unreadable(self.filed)
        self.counts = numpy.where(self.gaps, numpy.nan, self.filed)
        self.population = context(payload, "population", self.dates)
        self.rate = 1e5 * trailing_sum(self.counts) / self.population

    def at(self, values: NDArray[numpy.float64], period: str) -> float:
        """One month's value of a derived series, by 'MM-YYYY'."""
        month, year = (int(part) for part in period.split("-"))
        index = numpy.flatnonzero([date.year == year and date.month == month
                                   for date in self.dates])
        return float(values[index[0]]) if len(index) else float("nan")


arrested = {scope: ArrestSeries(scope, arrests[scope]) for scope in SCOPES}

# What the national total contains. The response's own twelve months of 2019,
# summed, against the FBI's estimate for the same year; then the share of that
# estimate belonging to the eight offences behind fbi_offenses' clearances.
national_dates, national_counts = monthly(arrests["national"])
summed_2019 = float(national_counts[[date.year == 2019 for date in national_dates]].sum())
estimated = CIUS_2019["all offences"]
clearable = CIUS_2019["violent crime"] + CIUS_2019["property crime"]
print(f"{'national arrests in 2019, this response summed':52} {summed_2019:>12,.0f}")
print(f"{'FBI estimate of 2019 arrests, all offences':52} {estimated:>12,}"
      f"  response {100 * (summed_2019 / estimated - 1):+.1f}%")
print(f"{'  for the eight offences with a clearance series':52} {clearable:>12,}"
      f"  {100 * clearable / estimated:.1f}% of the estimate")
print(f"the response's 2019 total is {summed_2019 / clearable:.1f} times the estimated arrests "
      f"for the offences\nwith a clearance series: roughly five arrests in six are for "
      f"offences fbi_offenses\ncannot meet")

YEARS = (1990, 2000, 2010, 2019, 2022, 2024)
print("\nArrests per 100,000 residents, any offence, twelve months to December")
print(f"{'place':16}" + "".join(f"{year:>9}" for year in YEARS)
      + f"{'readable months':>17}{'last window':>13}")
for record in arrested.values():
    readable = numpy.isfinite(record.counts) & (record.counts > 0)
    drawn = numpy.flatnonzero(numpy.isfinite(record.rate))
    print(f"{record.meta['name']:16}" + "".join(
        f"{record.at(record.rate, f'12-{year}'):9,.0f}"
        if numpy.isfinite(record.at(record.rate, f"12-{year}")) else f"{'—':>9}"
        for year in YEARS)
        + f"{int(readable.sum()):13d}/480{f'{record.dates[drawn[-1]]:%m-%Y}':>13}")


**The total is the all-offence total.** The national response's twelve months of
2019 sum to 10,316,114 arrests: 2.3% above the FBI's published estimate of
10,085,207 for every offence that year, and 6.6 times the 1,570,238 it estimated
for the eight offences that have a clearance series. Roughly five arrests in six
are for offences no clearance series in this API covers — the coverage point
above, measured. Why the response and the publication differ by 2.3%, neither
says.

**The levels.** In 1990 the four cities with a readable year arrested at 8,993
(Los Angeles), 9,086 (San Francisco), 11,212 (New York) and 12,417 (Washington)
per 100,000 residents, against 5,800 nationally. In 2024 the three that can be
read — New York at 2,747, Nashville at 2,809, Washington at 3,012 — sit close
together, against 2,070 nationally. Each of those years has its cities on one
system, so neither comparison sets one system's count against the other's. It is
still a comparison of totals over every offence, and each total carries its own
department's mix; the total alone cannot say how much of a difference is the
mix, which is why the other notebook goes and gets six of the offences.

**The long arc is a fall; the last three years are not, and are too short to
read.** Homicide bottomed out around 2013 and turned up after 2019 (section 2).
Arrests per resident show no recovery on that scale: the national rate is 5,800
in 1990, 4,910 in 2000, 4,355 in 2010 and 3,113 in 2019, and Los Angeles and
San Francisco — the two cities readable from 1990
to 2019 with no hole and no conversion between — go from 8,993 and 9,086 to
1,688 and 1,829. The national line has no hole either, but it is the sum of
whoever reported and so contains every agency conversion from 1991 on; "no
conversion" is not something it can have. Nashville is the exception, as it was
for homicide: 4,130 in 2000, then 14,374 in 2010 — more than three times any
other figure in that column — and 3,454 by 2019.

Since 2021 the column moves the other way, and the table above says so. Of the
three scopes with a readable year at both ends of it, Washington rises from
2,166 per 100,000 in the twelve months to December 2022 to 3,012 in 2024 and
Nashville from 2,119 to 2,809, while the nation is flat at 2,019 and 2,070; Los
Angeles and San Francisco have no readable 2024 to be read at. Two or three
years is not an arc, and for Washington it is the first stretch back from a
hole of twenty-nine years — the least trustworthy figure the department has.
[`enforcement.ipynb`](enforcement.ipynb) section 3 takes each scope's first
readable December from 2022 against its last, so that the two cities missing
from this column are included, and finds four rising, one falling and one flat.

**And the series breaks far harder than the homicide series.** New York has 222
readable months of 480 and Washington 113; the two hold one hole of twenty and a
half years and one of twenty-nine and a half, which is why their 1990 and 2024
figures have nothing in between. Los Angeles' and San Francisco's last readable
year ends in December 2023: San Francisco filed no arrest at all in 2024, in a
year its homicide counts are present. What each schedule looks like month by
month, and what the rule takes from each, is
[`enforcement.ipynb`](enforcement.ipynb) section 3.


## 4. Police strength — sworn officers per 100,000 residents

`fbi_employment` is **agency-grain only**: `pe/national` and `pe/state/{ST}`
answer 200 with the full response shape and every value null, so there is no
national line here and no honest way to manufacture one. Five calls, five
cities, annual since 1985.

Two choices in the arithmetic. The counts are male plus female sworn officers,
summed by the response model, because neither half alone is a series anybody
wants. And the denominator is the population from the *offence* response rather
than the "participated population" the employment payload carries — that one is
what the API's own per-1,000 rates divide by, and it is sometimes zero.

In [ ]:
staffing: dict[str, dict[int, float]] = {}
for scope in CITIES:
    payload = await call_tool("fbi_employment", {
        "ori": scope, "measure": "officers", "start": 1985, "end": 2024})
    staffing[scope] = {int(observation["period"]): float(observation["value"])
                       for observation in payload["observations"]
                       if observation["value"] is not None}
    for notice in payload["notices"]:
        print(f"{SCOPES[scope]['name']}: {notice}")


def per_100k(scope: str, year: int) -> float:
    """Sworn officers that year against the city's December population."""
    officers = staffing[scope].get(year)
    residents = places[scope].at(places[scope].population, f"12-{year}")
    return 1e5 * officers / residents if officers and numpy.isfinite(residents) else numpy.nan


years = (1985, 1990, 2000, 2010, 2019, 2024)
for title, value in (("sworn officers", lambda s, y: staffing[s].get(y, float("nan"))),
                     ("per 100,000 residents", per_100k)):
    print(f"\n{title}")
    print(f"{'place':16}" + "".join(f"{year:>9}" for year in years))
    for scope in CITIES:
        print(f"{SCOPES[scope]['name']:16}"
              + "".join(f"{value(scope, year):9,.0f}" for year in years))

Both halves of the question on one page: what each department closed, against
what it had to close it with. The staffing panel runs through 2021 without a
break — a headcount is filed annually and does not care which system the crimes
went into, which is the honest asymmetry between the two panels.

Every hole is on the left, and each is a filing schedule rather than a silent
department. Washington loses 1993, 1996, 1997 and 2021 — quarterly, twice,
once, and not at all. New York loses 1986, 1989 and 1995 to short clearance
gaps, 2002 to 2012 to its quarterly decade, and then 2021 and 2022 to the
two-year clearance gap above, so its line jumps straight from December 2020 to
December 2023. Los Angeles and San Francisco lose 2021 alone; the nation and
Nashville lose nothing.


In [ ]:
figure, (left, right) = pyplot.subplots(1, 2, figsize=(13.5, 5.5))
for axis in (left, right):
    axis.set_axisbelow(True)
    axis.axvspan(SRS_END, datetime(2022, 1, 1), color="0.55", alpha=0.25, lw=0)
    axis.set_xlabel("Year")
    axis.set_xlim(datetime(1985, 1, 1), datetime(2026, 1, 1))

# One point a year rather than every window: twelve monthly ratios a year, each
# sharing eleven months with the last, are noise drawn twelve times over.
for scope, place in places.items():
    years, annual_rate = place.annually(place.clearance)
    left.plot(years, annual_rate, color=place.meta["colour"], marker="o", markersize=3,
              lw=2.6 if scope == "national" else 1.6, label=place.meta["name"],
              zorder=2 if scope == "national" else 3)
left.axhline(100, color="0.3", lw=1, ls=":", zorder=1)
left.text(datetime(1986, 1, 1), 102, "every recorded homicide closed", fontsize=7.5, alpha=0.8)
left.set_ylim(0, 125)
left.set_ylabel("Homicides cleared, % of homicides recorded")
left.set_title("Clearance rate, the twelve months to each December", fontsize=11)

for scope in CITIES:
    reported = sorted(staffing[scope])
    # An annual headcount sits at mid-year rather than on the January boundary.
    right.plot([datetime(year, 7, 1) for year in reported],
               [per_100k(scope, year) for year in reported],
               color=SCOPES[scope]["colour"], lw=1.8, label=SCOPES[scope]["name"])
right.set_ylim(0, None)
right.set_ylabel("Sworn officers per 100,000 residents")
right.set_title("Police strength, annual — no national series exists", fontsize=11)

figure.tight_layout()
stamp_source("fbi_offenses(homicide, …, {clearances, offenses}, count) — ratio of trailing sums",
             "fbi_employment(…, officers) ÷ the population on the offence response",
             figure=figure)
# One legend for both panels, since the six colours mean the same in each.
handles = left.get_lines()[:len(places)]
figure.legend(handles, [handle.get_label() for handle in handles], loc="lower center",
              bbox_to_anchor=(0.5, 0.075), ncols=len(places), fontsize=9)
figure.subplots_adjust(bottom=0.23)

Read across the two panels and the sizes are the story. Washington employs
about 467 officers per 100,000 residents in 2024, twice Los Angeles' 232 and
San Francisco's 232, against a homicide rate five times theirs. The pairing
runs the opposite way to the intuition that more police means less crime, which
is exactly why it is a pairing and not a relation: cities staff to the crime
they have, and the causation runs both ways at once.

Within a city the movement is smaller and clearer. Four of the five are below
their 2019 strength — Washington by 14%, San Francisco by 10%, New York by 8%,
Los Angeles by 7% — while Nashville is the only one above it, 205 per 100,000
in 2019 against 220 in 2024, having added officers steadily since 1985. It is
also the city whose homicide rate rose most over the same years, which is worth
stating plainly as a fact about two series and not as anything else.

## 5. Does each city break at its own conversion date?

Here is the test the six scopes were chosen for.

The claim every FBI series carries as a notice is that **2021 is two breaks at
once**: agencies that had not converted stopped reporting, which is the
coverage dip; and NIBRS counts every offence in an incident where SRS counted
only the most serious, which changes what a count means.

Those two have different signatures, and the conversion calendar separates
them. A *reporting* break should land where the agency's own status put it — a
city already on NIBRS in 2020 has nothing to stop doing. A *counting* break
should land on the agency's own conversion date, 1999 for Nashville and 2024
for Los Angeles, and not in 2021 for a city that converted in another year; and
San Francisco, which never converted, should show one at no date at all.

The reporting half is settled by the table in section 2, restated here against
the dates.

In [ ]:
print(f"{'place':16} {'NIBRS from':12} {'2021 filed':>11}  what the registry date predicts")
for scope in CITIES:
    place = places[scope]
    filed = [date.month for date, gap in zip(place.dates, place.gaps)
             if date.year == 2021 and not gap]
    converted = (datetime.strptime(place.meta["nibrs"], "%Y-%m-%d")
                 if place.meta["nibrs"] else None)
    if converted and converted.year < 2021:
        verdict = "converted long before — nothing to stop"
    elif converted and converted.year == 2021:
        verdict = (f"gap should close in month {converted.month}; "
                   f"filing resumes in month {min(filed)}" if filed else "?")
    else:
        verdict = "not converted — nothing to file under"
    print(f"{place.meta['name']:16} {place.meta['nibrs'] or 'never':12} {len(filed):8d}/12  "
          f"{verdict}")

That is as clean as this kind of test gets. The one city that had converted
before 2021 is the one city with a complete 2021. Washington's hole closes in
the month the registry says it converted. The two cities that converted later
and the one that never did filed nothing at all. The reporting break is
**confirmed at monthly resolution against an outside date**, for all five.

It also shows something the notices do not. All four resumed filing *offences*
by January 2022, San Francisco included — and San Francisco has still never
converted, so whatever it files through is not NIBRS. 2021 was a one-year hole
in what the programme would accept, not the end of summary reporting.

**But the table above counts one leg, and the two legs did not come back
together everywhere.** Washington took both down and brought both back in
August 2021, which is its own conversion month. Los Angeles and San Francisco
brought both back in January 2022, a date that is neither city's conversion and
is simply where the hole ends. New York is the exception: its homicide counts
resume in January 2022 with the others, and its clearances do not resume until
January 2023 — twenty-four blank months rather than twelve — which is its
registry conversion date to the month.

That is the one thing in this notebook that lands on a conversion date and
looks like something. Take it for what it is. A department rebuilding its
reporting might well bring the offence leg up first and the clearance leg with
the conversion, and if that were the pattern it would be a finding about how
agencies migrate. It might equally be a department that lost two years of
clearance filing for reasons of its own and resumed in a January, as
departments do. One city, one date, and no way here to tell a mechanism from a
coincidence: the reporting break is confirmed at five cities, and this is a
hypothesis for a design with more of them in it.

The counting half needs a different offence, because homicide cannot test it:
it sits at the top of the hierarchy the old system applied, so the rule that
was abolished never suppressed one. **Aggravated assault** is the natural
probe. An assault committed during a robbery was counted only as the robbery
under SRS and is counted as both under NIBRS, so the hierarchy story predicts
assault counts step *up* at each city's own conversion date and nowhere else.

Six more calls, and two tests with them: each city against its own date, and
each city against 2021 — the second being a placebo, since a counting change
cannot reach a city that converted in 1999, in 2023, in 2024 or never.


In [ ]:
assault: dict[str, Payload] = {}
for scope in SCOPES:
    assault[scope] = await call_tool("fbi_offenses", {
        "offense": "aggravated-assault", "scope": scope, "measure": "offenses",
        "unit": "count", "start": START, "end": END})


def annual(payload: Payload) -> dict[int, tuple[float, int]]:
    """Calendar-year totals, and how many months of each were unreadable."""
    dates, counts = monthly(payload)
    gaps = unreadable(counts)
    totals: dict[int, tuple[float, int]] = {}
    for date, value, gap in zip(dates, counts, gaps):
        total, missing = totals.get(date.year, (0.0, 0))
        bad = bool(gap or numpy.isnan(value))
        totals[date.year] = (total + (0.0 if bad else float(value)), missing + int(bad))
    return totals


def step(totals: dict[int, tuple[float, int]], year: int, month: int) -> str:
    """Two clean years before a conversion against the two after it, as a change.

    A conversion mid-year makes that year neither, so it is skipped; a January
    conversion makes its own year the first year after.
    """
    before = (year - 2, year - 1)
    after = (year, year + 1) if month == 1 else (year + 1, year + 2)
    clean = lambda span: [totals[y][0] for y in span if y in totals and totals[y][1] == 0]
    show = lambda span: " ".join(
        f"{y}:" + ("—" if y not in totals else
                   f"{totals[y][0]:,.0f}" + ("*" if totals[y][1] else ""))
        for y in span)
    old, new = clean(before), clean(after)
    change = (f"{100 * (sum(new) / len(new)) / (sum(old) / len(old)) - 100:+6.0f}%"
              if old and new else "     —")
    return f"{show(before):>28}  {show(after):>28} {change}"


print("Aggravated assault, calendar-year counts either side of a date")
print(f"{'place':15} {'date':13} {'two years before':>28}  {'two years after':>28} change")
for scope in CITIES:
    totals = annual(assault[scope])
    if scope == "CA0380100":
        print(f"{SCOPES[scope]['name']:15} {'never':13} "
              f"{'no conversion to test against':>59}")
    else:
        converted = datetime.strptime(SCOPES[scope]["nibrs"], "%Y-%m-%d")
        print(f"{SCOPES[scope]['name']:15} {'its own, ' + converted.strftime('%Y-%m'):13} "
              + step(totals, converted.year, converted.month))
    if not (SCOPES[scope]["nibrs"] or "").startswith("2021"):
        print(f"{'':15} {'2021 placebo':13} " + step(totals, 2021, 1))
print("\n* a year with unreadable months, left out of the change")

# The control on the whole exercise: the largest steps in one of these series
# happen at dates no reporting system changed.
angeles = annual(assault["CA0194200"])
clean = [year for year in sorted(angeles) if angeles[year][1] == 0]
moves = sorted(((100 * (angeles[year][0] / angeles[year - 1][0] - 1), year)
                for year in clean[1:] if year - 1 in angeles and angeles[year - 1][1] == 0),
               key=lambda move: -abs(move[0]))
print("\nLos Angeles' largest year-on-year moves in aggravated assault, every month filed:")
for change, year in moves[:3]:
    print(f"  {year - 1} → {year}  {angeles[year - 1][0]:>8,.0f} → {angeles[year][0]:>8,.0f}"
          f"  {change:+6.0f}%")

**Inconclusive on the counting break, and informatively so.**

*Its own date.* Nashville, converting in October 1999, gives +5% over the two
years either side — the right sign, and small enough to be the ordinary drift
of a series that moves that much most years. New York gives −9% at its 2023
date and Washington −37% at its 2021 date: the wrong sign, and Washington's is
far too large for a counting rule and sits on top of its seven unfiled months.
Los Angeles converted in May 2024 and the data ends that December, so there is
no full year after it to compare — the test cannot be run at all.

*The placebo.* Every city that could not have had a counting change in 2021
shows a step across it anyway: New York +29%, Nashville +17%, San Francisco
+11%, Los Angeles +13%. Nashville converted twenty-one years earlier and San
Francisco never, so nothing about the hierarchy rule applies to either. Their
homicide counts move the same way over the same window — San Francisco +25%,
Los Angeles +27% — which is what a real rise in violence looks like, and it is
the rise everyone else was writing about in 2021.

So a step at 2021 is not evidence of the counting change, and the four steps at
the cities' own dates disagree in sign and size. **With one city per date and no
controls, this design cannot separate the counting rule from whatever else that
city did in those years** — a policy, a recording practice, a crime wave. Los
Angeles' own history is the standing reminder. Its three largest annual moves
in aggravated assault are +53% in 1986, −39% in 2005 and +39% in 2015 — every
month filed, all of them nineteen years or more from its conversion, and
nothing anywhere in the API marking any of them. They are larger than anything
2021 did to these cities, and a test that cannot tell them from a counting rule
is not measuring the counting rule.

What survives is the weaker, sounder claim. The reporting break is real, dated
and confirmed per city. The counting break is not measurable here, and for
homicide — the one offence the hierarchy rule never touched — the 2021
discontinuity in these series is a reporting discontinuity, which the coverage
arithmetic in section 2 removes.

## 6. What this supports, and what it does not

**Safe: within a city, over time — on this offence.** The forty-year arc in
section 2 is the strongest thing here: same department, same series, with the
filing gaps marked rather than smoothed. Washington at 81 per 100,000 in 1991
and 26 in 2024, or Nashville at 5.5 in 2013 and 14.7 in 2024, are real
statements about those cities. Both of those pairs straddle 2021, which the
third bullet below calls unsafe, so it is worth being explicit about why these
two survive the crossing. Homicide sits at the top of the SRS hierarchy: the
counting rule that was abolished in 2021 could not suppress one before or add
one after, so the definition is the same definition on both sides of the break.
And the reporting break removes windows rather than biasing the ones that
remain — an unfiled month voids every window it sits in instead of quietly
halving it, so a window that is drawn at all is drawn from twelve filed months.
Substitute any other offence and neither clause holds: the same 1991-to-2024
sentence about aggravated assault would be reading a definition change, a
filing gap and the city at once. The small-count floor in section 2 is the one
standing exception even here, and it is narrow: it changes no window on the
rate panel at all, and on the clearance panel only the five San Francisco years
section 3 names.

**Safe: between cities, in one year — because it is homicide, not because the
definitions agree.** Washington's 25.8 against San Francisco's 4.4 in 2024
holds up, and the population denominator comes back on the same response, so
the arithmetic is visible. What does *not* hold up is the reason one might give
for it. The six scopes were not filing to one system in 2024: Washington had
been on NIBRS since August 2021, New York since January 2023, Los Angeles since
May 2024 — five months of one system and seven of the other inside that single
year — and San Francisco has never converted at all. Three systems in one
year's cross-section. The comparison survives because the two systems agree
about homicide and about nothing else that matters here: the hierarchy rule
never reached the top of the hierarchy, so a 2024 homicide is the same event to
a summary reporter and an incident reporter. Read as a licence for homicide it
is right; read as a licence for cross-city comparison in general it is refuted
by section 5's own table, where Los Angeles crosses its conversion inside 2024
and nobody should set that year's assault count against San Francisco's.
Officers per 100,000 within a year is safe for a simpler reason: a headcount is
a headcount, whatever the crimes are filed under.

**Unsafe: levels across 2021.** Four of the five cities have no 2021 at all,
and the ones that converted changed what a count means while they were away. A
percentage change from 2019 to 2022 on these series is reading three things at
once. Homicide is the safest offence to try it on and it is still not clean.

**Unsafe: crime against staffing, as a relation.** Section 4 puts two panels
side by side and the paragraph under them is the whole of what can be said. The
cross-city correlation is positive and obviously not causal; nothing in this
API carries population composition, poverty, case mix or prosecution rates, and
a regression of homicides on officers using these two series would produce a
coefficient and no knowledge.

**Unsafe: a clearance rate read as a solve rate.** Section 3's definition is
the caveat. Clearances recorded in a window may close offences from earlier
years, which is how San Francisco reaches 94% and how four of these six pass
100%.

**Unsafe: arrests set against clearances, or read across a gap.** An arrest
counts a person on one occasion and a clearance an offence closed; the arrest
total runs across every offence, and by the FBI's 2019 estimate roughly five
arrests in six are for offences with no clearance series to meet; and an
exceptional clearance has no arrest in it at all. So the two neither difference
nor bound each other — [`enforcement.ipynb`](enforcement.ipynb) section 1
measures the same thing for one offence, where murder arrests outnumber homicide
clearances in 39 of the 40 years — and the split section 3 asks about, arrest
against exception, is served by none of the three tools here. Within a city the
arrest total is a count of occasions of every kind, and on its own it cannot say
which offences moved. And it breaks harder than the homicide series: New York
has 222 readable months of 480 and Washington 113, against the nation's 480 of
480.

Twenty-nine calls, three tools, six scopes. What this source gives that few
others do is all four measures at one grain from one API, with the population
and the coverage attached to every series — and, in the agency registry, an
outside date to test the data's own breaks against. What it does not give is
any way to know whether the break in front of you is in the world or in the
paperwork. That has to be gone and checked, as section 5 did, and sometimes the
answer is that it cannot be told.
